# Heart Disease Prediction Using Machine Learning
## Pluto Academy — Project 02

**Goal:** Train and compare three classification algorithms using the BRFSS 2020 Heart Disease dataset.

**Models:** Logistic Regression, Random Forest, and K-Nearest Neighbors (KNN).

**Dataset:** `heart_2020_cleaned.csv` — a cleaned subset of the 2020 U.S. CDC Behavioral Risk Factor Surveillance System (BRFSS).

> **Educational use only:** These are self-reported survey data from 2020, not current clinical records. This model is not a medical diagnostic tool and must not be used for healthcare decisions.


## 1. Import libraries and upload dataset
Run this cell and select `heart_2020_cleaned.csv` when Colab asks you to upload the file.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

if not os.path.exists("heart_2020_cleaned.csv"):
    uploaded = files.upload()
    filename = next(iter(uploaded))
    if filename != "heart_2020_cleaned.csv":
        os.rename(filename, "heart_2020_cleaned.csv")

df = pd.read_csv("heart_2020_cleaned.csv")
print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


## 2. Explore the dataset
Inspect data types, missing values, duplicate rows, and the target variable.

In [ ]:
print("Columns:", df.columns.tolist())
df.info()
print("\\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))
print("\\nDuplicate rows:", df.duplicated().sum())
print("\\nHeartDisease distribution:")
display(df["HeartDisease"].value_counts(dropna=False))


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="HeartDisease")
plt.title("Heart Disease Target Distribution")
plt.xlabel("Heart Disease")
plt.ylabel("Number of respondents")
plt.tight_layout()
plt.show()
display((df["HeartDisease"].value_counts(normalize=True) * 100).round(2))


## 3. Preprocessing and train/test split

- Remove exact duplicate rows and records with a missing target.
- Separate input features (`X`) from target (`y`).
- Split the data into 80% training and 20% testing, preserving class proportions.
- Impute missing values, scale numeric features, and one-hot encode categorical features inside a pipeline. This helps prevent data leakage.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

df = df.drop_duplicates().dropna(subset=["HeartDisease"]).copy()
X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"].astype(str).str.strip().str.title()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 4. Feature engineering

The dataset already provides demographic, lifestyle, and health-indicator variables. We retain these predictors, encode categorical variables, and scale numeric variables. After training, Random Forest feature importance will be used to identify influential model inputs. These importance scores do not imply causation.


## 5. Train three different models

The dataset is large, so to keep training practical in Google Colab on a mobile device, we use a reproducible stratified sample of up to 60,000 training rows and up to 20,000 test rows. The original 80/20 split is performed first; test records are not used for fitting. All three algorithms use the same sampled training and evaluation records.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Reproducible stratified samples for manageable training/evaluation time
if len(X_train) > 60000:
    X_train_model, _, y_train_model, _ = train_test_split(
        X_train, y_train, train_size=60000, random_state=42, stratify=y_train
    )
else:
    X_train_model, y_train_model = X_train, y_train

if len(X_test) > 20000:
    X_test_model, _, y_test_model, _ = train_test_split(
        X_test, y_test, train_size=20000, random_state=42, stratify=y_test
    )
else:
    X_test_model, y_test_model = X_test, y_test

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", solver="saga", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=120, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1, min_samples_leaf=2
    ),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1)
}

pipelines = {}
for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    print("Training:", name)
    pipe.fit(X_train_model, y_train_model)
    pipelines[name] = pipe

print("All three models trained.")
print("Training sample:", len(X_train_model), "| Evaluation sample:", len(X_test_model))


## 6. Evaluate and compare all models
Metrics: Accuracy, Precision, Recall, F1-score, and ROC-AUC. Since the target classes are imbalanced, do not judge models by accuracy alone.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

results = []
predictions = {}
probabilities = {}

for name, pipe in pipelines.items():
    pred = pipe.predict(X_test_model)
    predictions[name] = pred
    classes = list(pipe.named_steps["model"].classes_)
    if "Yes" in classes:
        prob_yes = pipe.predict_proba(X_test_model)[:, classes.index("Yes")]
        probabilities[name] = prob_yes
        auc = roc_auc_score((y_test_model == "Yes").astype(int), prob_yes)
    else:
        auc = np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_model, pred),
        "Precision": precision_score(y_test_model, pred, pos_label="Yes", zero_division=0),
        "Recall": recall_score(y_test_model, pred, pos_label="Yes", zero_division=0),
        "F1-score": f1_score(y_test_model, pred, pos_label="Yes", zero_division=0),
        "ROC-AUC": auc
    })

comparison = pd.DataFrame(results).sort_values("F1-score", ascending=False)
display(comparison.round(4))


In [ ]:
comparison.set_index("Model")[["Accuracy","Precision","Recall","F1-score","ROC-AUC"]].plot(
    kind="bar", figsize=(11,5), ylim=(0,1)
)
plt.title("Comparison of Three ML Models")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 7. Best model analysis and confusion matrix

The best model is selected by F1-score for this educational comparison. F1 balances precision and recall. In real screening applications, false negatives and false positives have different consequences and thresholds must be assessed by qualified domain experts.


In [ ]:
best_name = comparison.iloc[0]["Model"]
best_pred = predictions[best_name]
print("Best model by F1-score:", best_name)
print(classification_report(
    y_test_model, best_pred, labels=["No","Yes"], zero_division=0
))

cm = confusion_matrix(y_test_model, best_pred, labels=["No","Yes"])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted No","Predicted Yes"],
            yticklabels=["Actual No","Actual Yes"])
plt.title("Confusion Matrix — " + best_name)
plt.tight_layout()
plt.show()


## 8. Feature importance
Use Random Forest feature importance to rank the input variables after preprocessing.

In [ ]:
rf = pipelines["Random Forest"]
feature_names = rf.named_steps["preprocessor"].get_feature_names_out()
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.named_steps["model"].feature_importances_
}).sort_values("Importance", ascending=False).head(15)

display(importance)
plt.figure(figsize=(9,6))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Top 15 Random Forest Feature Importances")
plt.tight_layout()
plt.show()


## 9. Five-line conclusion
The cell below creates a conclusion using the actual metrics generated when you run the notebook.

In [ ]:
best = comparison.iloc[0]
print("CONCLUSION")
print("1. Logistic Regression, Random Forest, and KNN were trained and evaluated.")
print(f"2. {best['Model']} achieved the highest F1-score in this experiment.")
print(f"3. Its Accuracy={best['Accuracy']:.3f}, Precision={best['Precision']:.3f}, Recall={best['Recall']:.3f}.")
print("4. The confusion matrix summarizes correct and incorrect predictions for both classes.")
print("5. Findings apply only to this sampled 2020 self-reported survey dataset and are not clinical predictions.")


## 10. Save project outputs
Save the model comparison and feature-importance tables for GitHub submission.

In [ ]:
comparison.to_csv("model_comparison.csv", index=False)
importance.to_csv("feature_importance.csv", index=False)
print("Saved model_comparison.csv and feature_importance.csv")
files.download("model_comparison.csv")
files.download("feature_importance.csv")


## Dataset source and references

- **Cleaned dataset:** [Zenodo — BRFSS 2020 Heart Disease Dataset](https://zenodo.org/records/15364962)
- **Original survey documentation:** [CDC — 2020 BRFSS Survey Data and Documentation](https://www.cdc.gov/brfss/annual_data/annual_2020.html)

The CDC reports 401,958 records for the 2020 BRFSS survey. The cleaned Zenodo dataset selects 18 attributes for heart-disease analysis.

**Disclaimer:** This project is for educational machine-learning practice only. It is not a medical diagnostic system or a substitute for professional healthcare.
